Creating the webdriver

In [52]:
import email_info as ei
import smtplib
from email.message import EmailMessage
from email.utils import formataddr
import random
from json import JSONDecodeError

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
import hashlib
import json
from datetime import date, timedelta

In [53]:


url="https://jobs.lever.co/waabi"

chrome_options = webdriver.ChromeOptions()
chrome_options.add_experimental_option("detach", True)

driver=webdriver.Chrome(options=chrome_options)

driver.get(url=url)
driver.maximize_window()



In [54]:
with open("db.json","r") as f:
    try:
        db=json.load(f)
    except JSONDecodeError:
        db={}

# print(type(db))
# print(db.keys())

In [55]:
def get_text_or_none(parent, by, value):
    elems = parent.find_elements(by, value)
    return elems[0].text.strip() if elems else "unknown"

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:10]

#db={}


In [56]:
job_titles=driver.find_elements(By.CLASS_NAME,value="posting-title")
current_jobs=[]
new_job_alert=[]
old_job_filled_alert=[]

for index,job_title in enumerate(job_titles):
    url=job_title.get_attribute("href")
    job_id=sha256_hex(url)
    #print(f"{url}\t{job_id}")
    current_jobs.append(job_id)
    if job_id not in db.keys():
    # print(f"{url}\n{job_id}")
        db[job_id]=dict()
        # db[job_id]["index"]=index
        db[job_id]["job_name"]=get_text_or_none(job_title,By.CSS_SELECTOR,"h5")
        db[job_id]["work_policy"]=get_text_or_none(job_title,By.CSS_SELECTOR,".workplaceTypes")
        db[job_id]["work_policy"]=get_text_or_none(job_title,By.CSS_SELECTOR,".workplaceTypes")[:-2]
        db[job_id]["location"]=get_text_or_none(job_title,By.CSS_SELECTOR,".location")
        db[job_id]["commitment"]=get_text_or_none(job_title,By.CSS_SELECTOR,".commitment")
        db[job_id]["posted_date"]=date.today().strftime("%a %d-%b-%Y")
        db[job_id]["filled_date"]=""
        db[job_id]["url"]=url
        new_job_alert.append(job_id)

        # print(json.dumps(db[job_id], indent=4))


    # if index>3:
    #     break


In [57]:
for old_job in db.keys():
    if old_job not in current_jobs:
        db[old_job]["filled_date"]=(date.today() - timedelta(days=1)).strftime("%a %d-%b-%Y")
        old_job_filled_alert.append(old_job)

In [58]:
# print(json.dumps(db,indent=4))
print(f"{old_job_filled_alert= }\n{new_job_alert= }")

old_job_filled_alert= ['a97439cdf9c5']
new_job_alert= []


In [59]:
if old_job_filled_alert:
    info=""
    for filled_job in old_job_filled_alert:
        info+=db[filled_job]["job_name"]
        info+="\n"


    msg = EmailMessage()
    #msg["From"] = ei.email
    msg["From"]= formataddr(("Ghulam Samdani",ei.email))
    msg["To"] = ei.receivers_email
    msg["Subject"] = f"Waabi Filled Positions"
    msg.set_content(info)

    with smtplib.SMTP(ei.host_address,ei.port_address) as connection:
        connection.starttls()
        connection.login(user=ei.email,password=ei.password)
        connection.set_debuglevel(1)
        connection.send_message(msg)

send: 'mail FROM:<mughalsamdani2@gmail.com> size=267\r\n'
reply: b'250 2.1.0 OK af79cd13be357-8caf9fdf692sm359056985a.44 - gsmtp\r\n'
reply: retcode (250); Msg: b'2.1.0 OK af79cd13be357-8caf9fdf692sm359056985a.44 - gsmtp'
send: 'rcpt TO:<mughalsamdani1@gmail.com>\r\n'
reply: b'250 2.1.5 OK af79cd13be357-8caf9fdf692sm359056985a.44 - gsmtp\r\n'
reply: retcode (250); Msg: b'2.1.5 OK af79cd13be357-8caf9fdf692sm359056985a.44 - gsmtp'
send: 'data\r\n'
reply: b'354 Go ahead af79cd13be357-8caf9fdf692sm359056985a.44 - gsmtp\r\n'
reply: retcode (354); Msg: b'Go ahead af79cd13be357-8caf9fdf692sm359056985a.44 - gsmtp'
data: (354, b'Go ahead af79cd13be357-8caf9fdf692sm359056985a.44 - gsmtp')
send: b'From: Ghulam Samdani <mughalsamdani2@gmail.com>\r\nTo: mughalsamdani1@gmail.com\r\nSubject: Waabi Filled Positions\r\nContent-Type: text/plain; charset="utf-8"\r\nContent-Transfer-Encoding: 7bit\r\nMIME-Version: 1.0\r\n\r\nSystems Engineer, Platform Requirements and Verification\r\n.\r\n'
reply: b'250 2

In [60]:
for new_job in new_job_alert:
    #getting the jd
    driver.switch_to.new_window('tab')
    driver.get(url=url)

    info = ""
    descriptions = driver.find_elements(By.CLASS_NAME, value="section-wrapper")
    for desc in descriptions:
        info += desc.text


    msg = EmailMessage()
    #msg["From"] = ei.email
    msg["From"]= formataddr(("Ghulam Samdani",ei.email))
    msg["To"] = ei.receivers_email
    msg["Subject"] = f"Waabi {db[new_job]['job_name']}"
    msg.set_content(info)

    with smtplib.SMTP(ei.host_address,ei.port_address) as connection:
        connection.starttls()
        connection.login(user=ei.email,password=ei.password)
        connection.set_debuglevel(1)
        connection.send_message(msg)


In [61]:
with open("db.json","w") as f:
    json.dump(db,f,indent=4)
driver.quit()
